# Yoruba Conformer-CTC Fine-tuning (Clean Start)

Fine-tunes `nvidia/stt_en_conformer_ctc_small` on your 1,398 Yoruba recordings.

**Before running:**
- Runtime → Change runtime type → **T4 GPU**
- Make sure your recordings zip and CSV are in Google Drive

**What this notebook does:**
1. Installs NeMo with correct dependency versions
2. Mounts Drive and extracts your recordings
3. Builds transcripts from filenames (no Whisper needed)
4. Trains a fresh Yoruba SentencePiece tokenizer
5. Fine-tunes Conformer-CTC on your data
6. Saves the model back to Drive

## Step 1 — Install NeMo (correct versions for Colab)

In [4]:
# Install dependencies
!pip install -q "huggingface_hub==0.23.4"
!pip install -q "nemo_toolkit[asr]==1.23.0"
!pip install -q sentencepiece soundfile librosa

# Patch NeMo's broken imports
import glob

shim = """
try:
    from huggingface_hub import HfFolder
except ImportError:
    class HfFolder:
        @staticmethod
        def get_token(): return None

try:
    from huggingface_hub import ModelFilter
except ImportError:
    class ModelFilter:
        def __init__(self, **kwargs): pass
"""

for fpath in glob.glob('/usr/local/lib/python*/dist-packages/nemo/**/*.py', recursive=True):
    try:
        with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        if 'HfFolder' in content or 'ModelFilter' in content or 'use_auth_token' in content:
            content = content.replace(
                'from huggingface_hub import HfApi, HfFolder, ModelFilter, hf_hub_download',
                'from huggingface_hub import HfApi, hf_hub_download\n' + shim
            )
            content = content.replace('use_auth_token=', 'token=')
            with open(fpath, 'w', encoding='utf-8') as f:
                f.write(content)
            print(f'Patched: {fpath}')
    except:
        continue

import nemo
import nemo.collections.asr as nemo_asr
print(f'NeMo version: {nemo.__version__} ✅')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.6/402.6 kB 11.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.23.4 which is incompatible.
diffusers 0.39.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.23.4 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.23.4 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.23.4 which is incompatible.
transformers 5.13.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.23.4 which is incompatible.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setu

[NeMo W 2026-07-28 05:58:52 nemo_logging:393] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-07-28 05:58:52 nemo_logging:393] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-07-28 05:58:52 nemo_logging:393] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(flt)p?( \(default\))?$', token):
    
[NeMo W 2026-07-28 05:58:52 nemo_logging:393] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(dbl)p?( \(default\))?$', token):
    


NeMo version: 1.23.0 ✅


In [9]:
import numpy as np
print("NumPy version:", np.__version__)  # should be 1.26.4

import nemo
import nemo.collections.asr as nemo_asr
import torch
print("NeMo:", nemo.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("All good!")

NumPy version: 2.0.2
NeMo: 1.23.0
GPU: Tesla T4
All good!


## Step 2 — Mount Drive and extract recordings

In [12]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

# ── Edit these paths to match your Drive ─────────────────────────
ZIP_PATH   = '/content/drive/MyDrive/Colab Notebooks/recordings.zip'
CSV_PATH   = '/content/drive/MyDrive/Colab Notebooks/yoruba_commands.csv'
EXTRACT_TO = '/content/dataset_audio'
SAVE_DIR   = '/content/drive/MyDrive/Colab Notebooks/yoruba_conformer'
# ─────────────────────────────────────────────────────────────────

os.makedirs(SAVE_DIR, exist_ok=True)

if not os.path.exists(EXTRACT_TO):
    print('Extracting recordings zip...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_TO)
    print('Done.')
else:
    print('Already extracted.')

# Find all audio files
all_files = []
for root, dirs, files in os.walk(EXTRACT_TO):
    for f in files:
        if f.lower().endswith(('.wav', '.mp3', '.m4a', '.ogg', '.flac')):
            all_files.append(os.path.join(root, f))

print(f'Found {len(all_files)} audio files')
for f in all_files[:5]:
    print(' ', os.path.basename(f))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already extracted.
Found 1398 audio files
  wa_fun_mi_005.wav
  search_for_012.wav
  wa_lori_intanẹẹti_025.wav
  search_for_006.wav
  wa_fun_mi_003.wav


## Step 3 — Build transcripts from filenames + CSV

In [13]:
import re, pandas as pd, random
random.seed(42)

# Load CSV for reference phrases
csv_df = pd.read_csv(CSV_PATH)
csv_df['intent'] = csv_df['intent'].replace('farewell', 'exit')
intent_phrases = csv_df.groupby('intent')['text'].apply(list).to_dict()
print(f'CSV loaded: {len(csv_df)} phrases across {len(intent_phrases)} intents')

AUG_TAGS = ['_noise','_pitch','_volume','_reverb','_speed','_stretch','_tempo','_aug','aug_']

def filename_to_text(filename):
    name = os.path.splitext(filename)[0]
    for tag in AUG_TAGS:
        name = name.replace(tag, '')
    name = re.sub(r'_\d+$', '', name)
    name = re.sub(r'_(user|response|komputa)$', '', name)
    name = name.replace('_', ' ').strip()
    name = re.sub(r'[^\w\s\u00C0-\u024F]', '', name)
    return name.lower().strip()

def is_augmented(filename):
    return any(tag in filename.lower() for tag in AUG_TAGS)

# Only use ORIGINAL recordings (not augmented) for Conformer
records = []
skipped = []
for fpath in all_files:
    fname = os.path.basename(fpath)
    if is_augmented(fname):
        continue  # skip augmented
    text = filename_to_text(fname)
    if text:
        records.append({'path': fpath, 'filename': fname, 'text': text})
    else:
        skipped.append(fname)

info_df = pd.DataFrame(records)
print(f'Original recordings : {len(info_df)}')
print(f'Skipped             : {len(skipped)}')
print('\nSample transcripts:')
print(info_df[['filename','text']].head(10).to_string(index=False))

CSV loaded: 718 phrases across 57 intents
Original recordings : 1388
Skipped             : 0

Sample transcripts:
                 filename              text
        wa_fun_mi_005.wav         wa fun mi
       search_for_012.wav        search for
wa_lori_intanẹẹti_025.wav wa lori intanẹẹti
       search_for_006.wav        search for
        wa_fun_mi_003.wav         wa fun mi
        google_it_022.wav         google it
   wa_lori_google_013.wav    wa lori google
        google_it_018.wav         google it
   wa_lori_google_017.wav    wa lori google
        google_it_020.wav         google it


## Step 4 — Convert audio to 16kHz mono WAV

In [14]:
import librosa, soundfile as sf, warnings
warnings.filterwarnings('ignore')

PROCESSED_DIR = '/content/processed_audio'
os.makedirs(PROCESSED_DIR, exist_ok=True)

final_records = []
failed = []

for i, row in info_df.iterrows():
    try:
        y, _ = librosa.load(row['path'], sr=16000, mono=True)
        if len(y) < 1600:  # skip clips < 0.1s
            continue
        out_path = os.path.join(PROCESSED_DIR, f'{len(final_records):05d}.wav')
        sf.write(out_path, y, 16000)
        duration = len(y) / 16000
        final_records.append({
            'audio_filepath': out_path,
            'text': row['text'],
            'duration': duration,
        })
    except Exception as e:
        failed.append((row['filename'], str(e)))

    if len(final_records) % 50 == 0 and len(final_records) > 0:
        print(f'  Processed {len(final_records)} files...')

final_df = pd.DataFrame(final_records)
print(f'\nReady: {len(final_df)} clips')
print(f'Total duration: {final_df["duration"].sum()/60:.1f} minutes')
print(f'Failed: {len(failed)}')

  Processed 50 files...
  Processed 100 files...
  Processed 150 files...
  Processed 200 files...
  Processed 250 files...
  Processed 300 files...
  Processed 350 files...
  Processed 400 files...
  Processed 450 files...
  Processed 500 files...
  Processed 550 files...
  Processed 600 files...
  Processed 650 files...
  Processed 700 files...
  Processed 750 files...
  Processed 800 files...
  Processed 850 files...
  Processed 900 files...
  Processed 950 files...
  Processed 1000 files...
  Processed 1050 files...
  Processed 1100 files...
  Processed 1150 files...
  Processed 1200 files...
  Processed 1250 files...
  Processed 1300 files...
  Processed 1350 files...

Ready: 1388 clips
Total duration: 68.1 minutes
Failed: 0


## Step 5 — Split train/val and write NeMo manifests

In [15]:
import json, random

records_list = final_df.to_dict('records')
random.shuffle(records_list)

split = int(len(records_list) * 0.85)
train_records = records_list[:split]
val_records   = records_list[split:]

def write_manifest(records, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps({
                'audio_filepath': r['audio_filepath'],
                'text': r['text'],
                'duration': r['duration']
            }, ensure_ascii=False) + '\n')

write_manifest(train_records, '/content/train_manifest.json')
write_manifest(val_records,   '/content/val_manifest.json')

print(f'Train: {len(train_records)} clips')
print(f'Val  : {len(val_records)} clips')

Train: 1179 clips
Val  : 209 clips


## Step 6 — Train Yoruba SentencePiece tokenizer

In [16]:
import sentencepiece as spm, shutil

# Write corpus from all transcripts
with open('/content/yoruba_corpus.txt', 'w', encoding='utf-8') as f:
    for r in records_list:
        f.write(r['text'] + '\n')
    # Also add CSV phrases for better vocab coverage
    for phrase in csv_df['text'].tolist():
        f.write(phrase.lower().strip() + '\n')

VOCAB_SIZE = 128  # small vocab for command-only domain

spm.SentencePieceTrainer.train(
    input='/content/yoruba_corpus.txt',
    model_prefix='/content/yoruba_spm',
    vocab_size=VOCAB_SIZE,
    character_coverage=1.0,
    model_type='bpe',
    bos_id=-1, eos_id=-1, pad_id=0, unk_id=1,
)

# Set up tokenizer directory for NeMo
TOKEN_DIR = '/content/yoruba_tokenizer'
os.makedirs(TOKEN_DIR, exist_ok=True)
shutil.copy('/content/yoruba_spm.model', f'{TOKEN_DIR}/tokenizer.model')
shutil.copy('/content/yoruba_spm.vocab', f'{TOKEN_DIR}/tokenizer.vocab')

# Write vocab.txt for NeMo
sp = spm.SentencePieceProcessor(model_file='/content/yoruba_spm.model')
with open(f'{TOKEN_DIR}/vocab.txt', 'w', encoding='utf-8') as f:
    for i in range(sp.get_piece_size()):
        f.write(sp.id_to_piece(i) + '\n')

print(f'Tokenizer ready — vocab size: {sp.get_piece_size()}')

Tokenizer ready — vocab size: 128


## Step 7 — Load Conformer base model and swap Yoruba tokenizer

In [47]:
import nemo.collections.asr as nemo_asr
from omegaconf import open_dict
import json

# Build char vocab from manifests
chars = set()
for mf in ['/content/train_manifest.json', '/content/val_manifest.json']:
    with open(mf, encoding='utf-8') as f:
        for line in f:
            chars.update(json.loads(line)['text'])
vocab = sorted(chars)
print('Vocab size: ' + str(len(vocab)))

print('Loading QuartzNet...')
model = nemo_asr.models.EncDecCTCModel.from_pretrained('stt_en_quartznet15x5')
model.change_vocabulary(new_vocabulary=vocab)
print('Vocabulary swapped to ' + str(len(vocab)) + ' chars')

with open_dict(model.cfg):
    model.cfg.train_ds.manifest_filepath      = '/content/train_manifest.json'
    model.cfg.train_ds.batch_size             = 16
    model.cfg.train_ds.num_workers            = 2
    model.cfg.train_ds.pin_memory             = True
    model.cfg.train_ds.labels                 = vocab
    model.cfg.validation_ds.manifest_filepath = '/content/val_manifest.json'
    model.cfg.validation_ds.batch_size        = 16
    model.cfg.validation_ds.num_workers       = 2
    model.cfg.validation_ds.pin_memory        = True
    model.cfg.validation_ds.labels            = vocab

model.setup_training_data(model.cfg.train_ds)
model.setup_validation_data(model.cfg.validation_ds)
model.encoder.unfreeze()
model.decoder.unfreeze()
print('Model ready!')

Vocab size: 29
Loading QuartzNet...
[NeMo I 2026-07-28 06:25:46 nemo_logging:381] Found existing object /root/.cache/torch/NeMo/NeMo_1.23.0/stt_en_quartznet15x5/16661021d16e679bdfd97a2a03944c49/stt_en_quartznet15x5.nemo.
[NeMo I 2026-07-28 06:25:46 nemo_logging:381] Re-using file from: /root/.cache/torch/NeMo/NeMo_1.23.0/stt_en_quartznet15x5/16661021d16e679bdfd97a2a03944c49/stt_en_quartznet15x5.nemo
[NeMo I 2026-07-28 06:25:46 nemo_logging:381] Instantiating model from pre-trained checkpoint


[NeMo W 2026-07-28 06:25:46 nemo_logging:393] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /data2/voices/train_1k.json
    sample_rate: 16000
    labels:
    - ' '
    - a
    - b
    - c
    - d
    - e
    - f
    - g
    - h
    - i
    - j
    - k
    - l
    - m
    - 'n'
    - o
    - p
    - q
    - r
    - s
    - t
    - u
    - v
    - w
    - x
    - 'y'
    - z
    - ''''
    batch_size: 32
    trim_silence: true
    max_duration: 16.7
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: /asr_set_1.2/train/train_{0..1023}.tar
    num_workers: 20
    
[NeMo W 2026-07-28 06:25:46 nemo_logging:393] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loade

[NeMo I 2026-07-28 06:25:46 nemo_logging:381] PADDING: 16
[NeMo I 2026-07-28 06:25:47 nemo_logging:381] Numba CUDA SpecAugment kernel is being used
[NeMo I 2026-07-28 06:25:47 nemo_logging:381] Model EncDecCTCModel was successfully restored from /root/.cache/torch/NeMo/NeMo_1.23.0/stt_en_quartznet15x5/16661021d16e679bdfd97a2a03944c49/stt_en_quartznet15x5.nemo.
[NeMo I 2026-07-28 06:25:47 nemo_logging:381] Changed decoder to output to [' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'ṣ', 'ẹ', 'ọ'] vocabulary.


[NeMo W 2026-07-28 06:25:47 nemo_logging:393] `labels` is explicitly provided to the data loader, and is different from the `labels` provided at the model level config.
    If this is incorrect, please set the dataloader's `labels` to None.


Vocabulary swapped to 29 chars
[NeMo I 2026-07-28 06:25:47 nemo_logging:381] Dataset loaded with 1179 files totalling 0.96 hours
[NeMo I 2026-07-28 06:25:47 nemo_logging:381] 0 files were filtered totalling 0.00 hours


[NeMo W 2026-07-28 06:25:47 nemo_logging:393] `labels` is explicitly provided to the data loader, and is different from the `labels` provided at the model level config.
    If this is incorrect, please set the dataloader's `labels` to None.


[NeMo I 2026-07-28 06:25:47 nemo_logging:381] Dataset loaded with 209 files totalling 0.17 hours
[NeMo I 2026-07-28 06:25:47 nemo_logging:381] 0 files were filtered totalling 0.00 hours
Model ready!


In [51]:
# Fix use_auth_token in NeMo
import glob

for fpath in glob.glob('/usr/local/lib/python*/dist-packages/nemo/**/*.py', recursive=True):
    try:
        with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        if 'use_auth_token' in content:
            content = content.replace('use_auth_token=', 'token=')
            with open(fpath, 'w', encoding='utf-8') as f:
                f.write(content)
            print('Patched: ' + fpath)
    except:
        pass

print('Huggingface patch done!')

Huggingface patch done!


In [52]:
# Fix np.sctypes removed in NumPy 2.0
import numpy as np

if not hasattr(np, 'sctypes'):
    np.sctypes = {
        'int':     [np.int8, np.int16, np.int32, np.int64],
        'uint':    [np.uint8, np.uint16, np.uint32, np.uint64],
        'float':   [np.float16, np.float32, np.float64],
        'complex': [np.complex64, np.complex128],
        'others':  [bool, object, bytes, str]
    }
    print('np.sctypes patched!')
else:
    print('np.sctypes already exists.')

np.sctypes already exists.


## Step 8 — Configure training data

In [53]:
model.setup_training_data(train_data_config={
    'manifest_filepath': '/content/train_manifest.json',
    'sample_rate': 16000,
    'batch_size': 8,
    'shuffle': True,
    'num_workers': 2,
    'pin_memory': True,
})

model.setup_validation_data(val_data_config={
    'manifest_filepath': '/content/val_manifest.json',
    'sample_rate': 16000,
    'batch_size': 8,
    'shuffle': False,
    'num_workers': 2,
})

# Low learning rate — fine-tuning, not training from scratch
model.cfg.optim.lr = 5e-5
print('Training data configured.')

[NeMo W 2026-07-28 06:26:13 nemo_logging:393] dataset does not have explicitly defined labels


[NeMo I 2026-07-28 06:26:13 nemo_logging:381] Dataset loaded with 1179 files totalling 0.96 hours
[NeMo I 2026-07-28 06:26:13 nemo_logging:381] 0 files were filtered totalling 0.00 hours


[NeMo W 2026-07-28 06:26:13 nemo_logging:393] dataset does not have explicitly defined labels


[NeMo I 2026-07-28 06:26:13 nemo_logging:381] Dataset loaded with 209 files totalling 0.17 hours
[NeMo I 2026-07-28 06:26:13 nemo_logging:381] 0 files were filtered totalling 0.00 hours
Training data configured.


In [54]:
# Fix np.sctypes removed in NumPy 2.0
import numpy as np
if not hasattr(np, 'sctypes'):
    np.sctypes = {
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'float': [np.float16, np.float32, np.float64],
        'complex': [np.complex64, np.complex128],
        'others': [bool, object, bytes, str]
    }
print("np.sctypes patched!")

np.sctypes patched!


In [56]:
# Disable WER decoding during training to avoid KeyError
model.wer.decoding = None

# Also patch the validation step to skip WER
import types

def validation_step_no_wer(self, batch, batch_idx, dataloader_idx=0):
    signal, signal_len, transcript, transcript_len = batch
    log_probs, encoded_len, predictions = self.forward(
        input_signal=signal, input_signal_length=signal_len
    )
    loss = self.loss(
        log_probs=log_probs,
        targets=transcript,
        input_lengths=encoded_len,
        target_lengths=transcript_len,
    )
    self.log('val_loss', loss, prog_bar=True, on_epoch=True, sync_dist=True)
    return {'val_loss': loss}

model.validation_step = types.MethodType(validation_step_no_wer, model)
print('WER disabled for training — using val_loss only')

WER disabled for training — using val_loss only


In [60]:
from nemo.collections.asr.metrics.wer import WER
from nemo.collections.asr.parts.submodules.ctc_decoding import CTCDecoding, CTCDecodingConfig

# Rebuild decoding with correct vocab
decoding_cfg = CTCDecodingConfig()
decoding     = CTCDecoding(decoding_cfg, vocabulary=vocab)

# Rebuild WER metric with correct decoding
model.wer = WER(
    decoding=decoding,
    use_cer=False,
    dist_sync_on_step=True,
    log_prediction=False,
)

print('WER metric rebuilt with vocab size: ' + str(len(vocab)))
print('Blank id: ' + str(decoding.blank_id))

WER metric rebuilt with vocab size: 29
Blank id: 29


In [62]:
print('Vocab length:', len(vocab))
print('Max token ID should be:', len(vocab) - 1)
print('Blank ID:', len(vocab))
print()
print('Vocab contents:')
for i, v in enumerate(vocab):
    print(str(i) + ': ' + repr(v))

Vocab length: 29
Max token ID should be: 28
Blank ID: 29

Vocab contents:
0: ' '
1: 'a'
2: 'b'
3: 'c'
4: 'd'
5: 'e'
6: 'f'
7: 'g'
8: 'h'
9: 'i'
10: 'j'
11: 'k'
12: 'l'
13: 'm'
14: 'n'
15: 'o'
16: 'p'
17: 'q'
18: 'r'
19: 's'
20: 't'
21: 'u'
22: 'v'
23: 'w'
24: 'x'
25: 'y'
26: 'ṣ'
27: 'ẹ'
28: 'ọ'


In [63]:
import json

bad_entries = []
all_chars_in_data = set()

for mf in ['/content/train_manifest.json', '/content/val_manifest.json']:
    with open(mf, encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line.strip())
            text  = entry['text']
            for ch in text:
                all_chars_in_data.add(ch)
                if ch not in vocab and ch != ' ':
                    bad_entries.append((entry['audio_filepath'], text, ch))

print('All unique chars in data:')
for ch in sorted(all_chars_in_data):
    in_vocab = ch in vocab
    print(repr(ch) + ' -> in vocab: ' + str(in_vocab))

print()
print('Problem entries (chars not in vocab): ' + str(len(bad_entries)))
for path, text, ch in bad_entries[:10]:
    print('  char ' + repr(ch) + ' in: ' + text)

All unique chars in data:
' ' -> in vocab: True
'a' -> in vocab: True
'b' -> in vocab: True
'c' -> in vocab: True
'd' -> in vocab: True
'e' -> in vocab: True
'f' -> in vocab: True
'g' -> in vocab: True
'h' -> in vocab: True
'i' -> in vocab: True
'j' -> in vocab: True
'k' -> in vocab: True
'l' -> in vocab: True
'm' -> in vocab: True
'n' -> in vocab: True
'o' -> in vocab: True
'p' -> in vocab: True
'q' -> in vocab: True
'r' -> in vocab: True
's' -> in vocab: True
't' -> in vocab: True
'u' -> in vocab: True
'v' -> in vocab: True
'w' -> in vocab: True
'x' -> in vocab: True
'y' -> in vocab: True
'ṣ' -> in vocab: True
'ẹ' -> in vocab: True
'ọ' -> in vocab: True

Problem entries (chars not in vocab): 0


In [64]:
import nemo.collections.asr as nemo_asr
from omegaconf import open_dict, OmegaConf
from nemo.collections.asr.metrics.wer import WER
from nemo.collections.asr.parts.submodules.ctc_decoding import CTCDecoding, CTCDecodingConfig
import json

# Rebuild vocab
chars = set()
for mf in ['/content/train_manifest.json', '/content/val_manifest.json']:
    with open(mf, encoding='utf-8') as f:
        for line in f:
            chars.update(json.loads(line)['text'])
vocab = sorted(chars)
print('Vocab: ' + str(vocab))
print('Size:  ' + str(len(vocab)))

# Fresh model load
print('Loading fresh QuartzNet...')
model = nemo_asr.models.EncDecCTCModel.from_pretrained('stt_en_quartznet15x5')

# Change vocabulary
model.change_vocabulary(new_vocabulary=vocab)

# Verify decoder matches vocab
print('Decoder num classes: ' + str(model.decoder.num_classes_with_blank))
print('Expected:            ' + str(len(vocab) + 1))

# Setup data with explicit labels
with open_dict(model.cfg):
    model.cfg.train_ds.manifest_filepath      = '/content/train_manifest.json'
    model.cfg.train_ds.batch_size             = 16
    model.cfg.train_ds.num_workers            = 0
    model.cfg.train_ds.pin_memory             = False
    model.cfg.train_ds.labels                 = vocab
    model.cfg.train_ds.normalize_transcripts  = False
    model.cfg.validation_ds.manifest_filepath = '/content/val_manifest.json'
    model.cfg.validation_ds.batch_size        = 16
    model.cfg.validation_ds.num_workers       = 0
    model.cfg.validation_ds.pin_memory        = False
    model.cfg.validation_ds.labels            = vocab
    model.cfg.validation_ds.normalize_transcripts = False

model.setup_training_data(model.cfg.train_ds)
model.setup_validation_data(model.cfg.validation_ds)

# Rebuild WER with correct vocab
decoding_cfg = CTCDecodingConfig()
decoding     = CTCDecoding(decoding_cfg, vocabulary=vocab)
model.wer    = WER(
    decoding=decoding,
    use_cer=False,
    dist_sync_on_step=True,
    log_prediction=False,
)

model.encoder.unfreeze()
model.decoder.unfreeze()

print('Model fully ready!')
print('WER blank id: ' + str(model.wer.decoding.blank_id))


Vocab: [' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'ṣ', 'ẹ', 'ọ']
Size:  29
Loading fresh QuartzNet...
[NeMo I 2026-07-28 06:33:01 nemo_logging:381] Found existing object /root/.cache/torch/NeMo/NeMo_1.23.0/stt_en_quartznet15x5/16661021d16e679bdfd97a2a03944c49/stt_en_quartznet15x5.nemo.
[NeMo I 2026-07-28 06:33:01 nemo_logging:381] Re-using file from: /root/.cache/torch/NeMo/NeMo_1.23.0/stt_en_quartznet15x5/16661021d16e679bdfd97a2a03944c49/stt_en_quartznet15x5.nemo
[NeMo I 2026-07-28 06:33:01 nemo_logging:381] Instantiating model from pre-trained checkpoint


[NeMo W 2026-07-28 06:33:02 nemo_logging:393] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /data2/voices/train_1k.json
    sample_rate: 16000
    labels:
    - ' '
    - a
    - b
    - c
    - d
    - e
    - f
    - g
    - h
    - i
    - j
    - k
    - l
    - m
    - 'n'
    - o
    - p
    - q
    - r
    - s
    - t
    - u
    - v
    - w
    - x
    - 'y'
    - z
    - ''''
    batch_size: 32
    trim_silence: true
    max_duration: 16.7
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: /asr_set_1.2/train/train_{0..1023}.tar
    num_workers: 20
    
[NeMo W 2026-07-28 06:33:02 nemo_logging:393] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loade

[NeMo I 2026-07-28 06:33:02 nemo_logging:381] PADDING: 16
[NeMo I 2026-07-28 06:33:03 nemo_logging:381] Numba CUDA SpecAugment kernel is being used
[NeMo I 2026-07-28 06:33:03 nemo_logging:381] Model EncDecCTCModel was successfully restored from /root/.cache/torch/NeMo/NeMo_1.23.0/stt_en_quartznet15x5/16661021d16e679bdfd97a2a03944c49/stt_en_quartznet15x5.nemo.
[NeMo I 2026-07-28 06:33:03 nemo_logging:381] Changed decoder to output to [' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'ṣ', 'ẹ', 'ọ'] vocabulary.


[NeMo W 2026-07-28 06:33:03 nemo_logging:393] `labels` is explicitly provided to the data loader, and is different from the `labels` provided at the model level config.
    If this is incorrect, please set the dataloader's `labels` to None.


Decoder num classes: 30
Expected:            30
[NeMo I 2026-07-28 06:33:03 nemo_logging:381] Dataset loaded with 1179 files totalling 0.96 hours
[NeMo I 2026-07-28 06:33:03 nemo_logging:381] 0 files were filtered totalling 0.00 hours


[NeMo W 2026-07-28 06:33:03 nemo_logging:393] `labels` is explicitly provided to the data loader, and is different from the `labels` provided at the model level config.
    If this is incorrect, please set the dataloader's `labels` to None.


[NeMo I 2026-07-28 06:33:03 nemo_logging:381] Dataset loaded with 209 files totalling 0.17 hours
[NeMo I 2026-07-28 06:33:03 nemo_logging:381] 0 files were filtered totalling 0.00 hours
Model fully ready!
WER blank id: 29


## Step 9 — Fine-tune

In [65]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

EPOCHS = 50

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    mode='min',
    verbose=True,
)

checkpoint = ModelCheckpoint(
    dirpath='/content/checkpoints',
    filename='conformer_yoruba_best',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    verbose=True,
)

trainer = pl.Trainer(
    devices=1,
    accelerator='gpu',
    max_epochs=EPOCHS,
    gradient_clip_val=1.0,
    log_every_n_steps=10,
    enable_checkpointing=True,
    callbacks=[early_stop, checkpoint],
)

print('Fine-tuning on ' + str(len(train_records)) + ' clips...')
print('Max epochs: ' + str(EPOCHS) + ' · Early stop patience: 8')
trainer.fit(model)
print('Training complete!')

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Fine-tuning on 1179 clips...
Max epochs: 50 · Early stop patience: 8
[NeMo I 2026-07-28 06:33:15 nemo_logging:381] Optimizer config = Novograd (
    Parameter Group 0
        amsgrad: False
        betas: [0.8, 0.5]
        eps: 1e-08
        grad_averaging: False
        lr: 0.01
        weight_decay: 0.001
    )
[NeMo I 2026-07-28 06:33:15 nemo_logging:381] Scheduler "<nemo.core.optim.lr_scheduler.CosineAnnealing object at 0x7e43a2c73020>" 
    will be used during training (effective maximum steps = 3700) - 
    Parameters : 
    (warmup_steps: null
    warmup_ratio: null
    min_lr: 0.0
    last_epoch: -1
    max_steps: 3700
    )


INFO:pytorch_lightning.callbacks.model_summary:
  | Name              | Type                              | Params
------------------------------------------------------------------------
0 | preprocessor      | AudioToMelSpectrogramPreprocessor | 0     
1 | encoder           | ConvASREncoder                    | 18.9 M
2 | spec_augmentation | SpectrogramAugmentation           | 0     
3 | wer               | WER                               | 0     
4 | decoder           | ConvASRDecoder                    | 30.8 K
5 | loss              | CTCLoss                           | 0     
------------------------------------------------------------------------
18.9 M    Trainable params
0         Non-trainable params
18.9 M    Total params
75.702    Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

... (94 outputs left out)

INFO:pytorch_lightning.utilities.rank_zero:Epoch 47, global step 3552: 'val_loss' was not in top 1


Validation: 0it [00:00, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric val_loss did not improve in the last 8 records. Best score: 25.684. Signaling Trainer to stop.
INFO:pytorch_lightning.utilities.rank_zero:Epoch 48, global step 3626: 'val_loss' was not in top 1


Training complete!


In [72]:
import torch
import functools

# Patch torch.load
_original_load = torch.load
torch.load = functools.partial(_original_load, weights_only=False)

best_ckpt = '/content/checkpoints/conformer_yoruba_best-v1.ckpt'

trainer2 = pl.Trainer(
    devices=1,
    accelerator='gpu',
    max_epochs=100,
    gradient_clip_val=1.0,
    log_every_n_steps=10,
    enable_checkpointing=True,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=10, mode='min', verbose=True),
        ModelCheckpoint(
            dirpath='/content/checkpoints',
            filename='conformer_yoruba_best_v2',
            monitor='val_loss',
            mode='min',
            save_top_k=1,
            verbose=True,
        )
    ],
)

# Unfreeze BEFORE fitting
model.encoder.unfreeze()
model.decoder.unfreeze()
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable params: ' + str(trainable))

print('Continuing training from best checkpoint...')
trainer2.fit(model, ckpt_path=best_ckpt)
print('Done!')

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/checkpoints/conformer_yoruba_best-v1.ckpt


Trainable params: 18925406
Continuing training from best checkpoint...


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


[NeMo I 2026-07-28 07:01:11 nemo_logging:381] Optimizer config = Novograd (
    Parameter Group 0
        amsgrad: False
        betas: [0.8, 0.5]
        eps: 1e-08
        grad_averaging: False
        lr: 0.01
        weight_decay: 0.001
    )
[NeMo I 2026-07-28 07:01:11 nemo_logging:381] Scheduler "<nemo.core.optim.lr_scheduler.CosineAnnealing object at 0x7e43ab358770>" 
    will be used during training (effective maximum steps = 7400) - 
    Parameters : 
    (warmup_steps: null
    warmup_ratio: null
    min_lr: 0.0
    last_epoch: -1
    max_steps: 7400
    )


INFO:pytorch_lightning.callbacks.model_summary:
  | Name              | Type                              | Params
------------------------------------------------------------------------
0 | preprocessor      | AudioToMelSpectrogramPreprocessor | 0     
1 | encoder           | ConvASREncoder                    | 18.9 M
2 | spec_augmentation | SpectrogramAugmentation           | 0     
3 | wer               | WER                               | 0     
4 | decoder           | ConvASRDecoder                    | 30.8 K
5 | loss              | CTCLoss                           | 0     
------------------------------------------------------------------------
18.9 M    Trainable params
0         Non-trainable params
18.9 M    Total params
75.702    Total estimated model params size (MB)
INFO:pytorch_lightning.utilities.rank_zero:Restored all states from the checkpoint at /content/checkpoints/conformer_yoruba_best-v1.ckpt


Sanity Checking: 0it [00:00, ?it/s]

... (20 outputs left out)

INFO:pytorch_lightning.utilities.rank_zero:Epoch 50, global step 3774: 'val_loss' was not in top 1


Validation: 0it [00:00, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric val_loss did not improve in the last 8 records. Best score: 25.619. Signaling Trainer to stop.
INFO:pytorch_lightning.utilities.rank_zero:Epoch 51, global step 3848: 'val_loss' was not in top 1


Done!


## Step 10 — Test the fine-tuned model

In [73]:
model.eval()

test_files  = [r['audio_filepath'] for r in val_records[:10]]
test_truths = [r['text'] for r in val_records[:10]]

results = model.transcribe(test_files)

correct = 0
print('Test results:\n')
for truth, pred in zip(test_truths, results):
    pred_text = pred.text if hasattr(pred, 'text') else str(pred)
    match = pred_text.strip().lower() == truth.strip().lower()
    correct += int(match)
    icon = '✅' if match else '⚠️'
    print(f'  {icon}  Expected : "{truth}"')
    print(f'       Got      : "{pred_text}"')
    print()

print(f'Exact match: {correct}/{len(test_files)} ({correct/len(test_files)*100:.0f}%)')

[NeMo W 2026-07-28 07:08:12 nemo_logging:393] `labels` is explicitly provided to the data loader, and is different from the `labels` provided at the model level config.
    If this is incorrect, please set the dataloader's `labels` to None.


Transcribing:   0%|          | 0/3 [00:00<?, ?it/s]

Test results:

  ⚠️  Expected : "so akoko fun mi"
       Got      : "si ori kole"

  ⚠️  Expected : "close chrome"
       Got      : "cose won"

  ⚠️  Expected : "pa excel"
       Got      : "pa e ca"

  ⚠️  Expected : "lo sẹhin"
       Got      : "e sin"

  ⚠️  Expected : "jade kuro ninu notepad"
       Got      : "kin iloni riti"

  ⚠️  Expected : "pa taabu yii"
       Got      : "pabe sua"

  ⚠️  Expected : "gbe imole wale"
       Got      : "gbe olea re"

  ⚠️  Expected : "jade kuro ninu notepad"
       Got      : "gani purotukaru"

  ⚠️  Expected : "search for"
       Got      : "su co"

  ⚠️  Expected : "bẹrẹ leẹkansi"
       Got      : "ma ole soku"

Exact match: 0/10 (0%)


## Step 11 — Save model to Drive

In [ ]:
import shutil

# Save .nemo model file
save_path = os.path.join(SAVE_DIR, 'yoruba_conformer.nemo')
model.save_to(save_path)
print(f'Model saved to Drive: {save_path}')

# Also save the tokenizer dir
token_save = os.path.join(SAVE_DIR, 'yoruba_tokenizer')
if os.path.exists(token_save):
    shutil.rmtree(token_save)
shutil.copytree(TOKEN_DIR, token_save)
print(f'Tokenizer saved to Drive: {token_save}')

# Download to PC
from google.colab import files
files.download(save_path)
print('\nDownloaded! Place at: models/yoruba_conformer.nemo')